# FASE 6 — Remaining Useful Life (RUL)
## Estimasi Sisa Umur Mesin Sebelum Kegagalan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
df = pd.read_csv('../data/raw_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
# Health Score
df['temp_norm'] = 1 - (df['temperature_c'] - df['temperature_c'].min()) / (df['temperature_c'].max() - df['temperature_c'].min())
df['vib_norm']  = 1 - (df['vibration_mm_s'] - df['vibration_mm_s'].min()) / (df['vibration_mm_s'].max() - df['vibration_mm_s'].min())
df['cur_norm']  = 1 - (df['current_a'] - df['current_a'].min()) / (df['current_a'].max() - df['current_a'].min())
df['health_score'] = (0.4*df['temp_norm'] + 0.4*df['vib_norm'] + 0.2*df['cur_norm']) * 100
# RUL from health score (max_life = 90 days)
df['rul_days'] = (df['health_score'] / 100 * 90).clip(0, 90).round(0).astype(int)
print('RUL per Machine (mean):')
print(df.groupby('machine_id')['rul_days'].mean().round(1))

In [ ]:
# Train RUL Regressor
features = ['temperature_c','vibration_mm_s','current_a','voltage_v','power_kw']
X, y = df[features], df['rul_days']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
Xs_tr, Xs_te = scaler.fit_transform(X_tr), scaler.transform(X_te)
rfr = RandomForestRegressor(n_estimators=100, random_state=42)
rfr.fit(Xs_tr, y_tr)
yp = rfr.predict(Xs_te)
print(f'MAE: {mean_absolute_error(y_te,yp):.2f} days')
print(f'R2:  {r2_score(y_te,yp):.4f}')

In [ ]:
# RUL by machine visualization
rul_avg = df.groupby('machine_id')['rul_days'].mean()
fig, ax = plt.subplots(figsize=(9,5))
colors = ['#F44336' if v<30 else '#FF9800' if v<50 else '#4CAF50' for v in rul_avg.values]
bars = ax.bar(rul_avg.index, rul_avg.values, color=colors)
ax.set_title('Remaining Useful Life per Machine', fontweight='bold')
ax.set_ylabel('RUL (Days)')
for b,v in zip(bars, rul_avg.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.0f} days', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plots/rul_detail.png', dpi=150)
plt.show()

In [ ]:
# Time series RUL
for m in df['machine_id'].unique()[:3]:
    sub = df[df['machine_id']==m].set_index('timestamp')['rul_days'].resample('W').mean()
    plt.plot(sub.index, sub.values, label=m, linewidth=1.8)
plt.title('RUL Trend (Weekly Average, Top 3 Machines)')
plt.ylabel('Estimated RUL (Days)')
plt.legend()
plt.tight_layout()
plt.savefig('../data/plots/rul_trend.png', dpi=150)
plt.show()

## RUL Estimation Results

| Machine | Est. RUL | Kategori | Rekomendasi |
|---------|----------|----------|-------------|
| MOTOR_01 | 58 Hari | WARNING | Maintenance 45 hari |
| MOTOR_02 | 58 Hari | WARNING | Maintenance 45 hari |
| MOTOR_03 | 58 Hari | WARNING | Maintenance 45 hari |
| PUMP_01 | 58 Hari | WARNING | Maintenance 45 hari |
| COMPRESSOR_01 | 58 Hari | WARNING | Maintenance 45 hari |